# Dataset Discovery Notebook

Guided exploration of a new radiological dataset to gather all information needed
for the `add-dataset` skill. Run each section, fill in values where prompted,
and the final cell exports a complete `configs/datasets/<registry_key>.json`.

---

## How to use

1. Fill in the **User Input** cells (marked with `# >>> EDIT THIS`).
2. Run each section — the code will inspect your CSVs, directories, and (optionally) DICOM headers.
3. Review the auto-discovered information and override anything incorrect.
4. The final cell writes the JSON config file.

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

import pandas as pd

---

## 1. Basic Dataset Info

Fill in the basic metadata about the dataset.

**`base_image_dir` convention** — set `BASE_IMAGE_DIR` to the **deepest stable directory whose immediate contents are the entries named in `image_path`**:

- For Train/Test split datasets, point at the split-specific image dir (e.g. `train_images/`, `test_images/`, `dicom-images-train/`), **not** the competition root.
- If the upstream CSV stores paths with a prefix the user shouldn't have to provide (e.g. CheXpert's `Path` column starts with `CheXpert-v1.0/train/...`, MIMIC-CXR's `path` starts with `files/...`), strip that prefix in `_build_image_path` so `BASE_IMAGE_DIR` aligns with the deepest stable directory.
- **Structural exception**: if the images are split across multiple sibling subdirs at the same level (e.g. ChestX-ray14's `images_001/` … `images_012/`), `BASE_IMAGE_DIR` is the parent and `image_path` carries the subdir name. Document the exception in the harmonizer's docstring.

See `.claude/skills/add-dataset/SKILL.md` and the wiki for the full convention.

In [ ]:
# >>> EDIT THIS — basic dataset metadata
DATASET_NAME = "ChestX-ray14"  # e.g. "MIMIC-CXR", "ChestX-ray14"
REGISTRY_KEY = "chestxray14"  # e.g. "mimic_cxr", "chestxray14" (snake_case)
DESCRIPTION = "CXR dataset which has nlp derived labels and a subset has bboxes"  # free-text description of the dataset
MODALITY = "2D"  # "2D" or "3D"

# >>> EDIT THIS — deepest stable directory whose immediate contents are
# the entries named in image_path. For split datasets, the split-specific
# image dir (e.g. train_images/), NOT the competition root.
BASE_IMAGE_DIR = "/path/to/NIH_CXR/CXR14/"  # e.g. ".../mimic-cxr/2.1.0/files/"

# >>> EDIT THIS — path to the primary metadata CSV
PRIMARY_CSV_PATH = "/path/to/NIH_CXR/CXR14/Data_Entry_2017.csv"  # e.g. "/path/to/.../cxr-record-list.csv.gz"

# >>> EDIT THIS (optional) — path to a separate label CSV, or "" if labels are in primary CSV
LABEL_CSV_PATH = "/path/to/NIH_CXR/CXR14/Data_Entry_2017.csv"

# >>> EDIT THIS (optional) — other CSVs (reports, view position, masks, bboxes)
REPORT_CSV_PATH = ""
VIEW_POSITION_CSV_PATH = "/path/to/NIH_CXR/CXR14/Data_Entry_2017.csv"
MASK_CSV_PATH = ""
BBOX_CSV_PATH = "/path/to/NIH_CXR/CXR14/BBox_List_2017.csv"

---

## 2. Explore the Image Directory

Discover the image file format and directory structure.

In [ ]:
if BASE_IMAGE_DIR:
    base = Path(BASE_IMAGE_DIR)
    assert base.is_dir(), f"Directory not found: {BASE_IMAGE_DIR}"

    # Top-level contents
    top_items = sorted(base.iterdir())
    print(f"Top-level items in base_image_dir ({len(top_items)}):")
    for item in top_items[:20]:
        kind = "DIR " if item.is_dir() else "FILE"
        print(f"  {kind}  {item.name}")
    if len(top_items) > 20:
        print(f"  ... and {len(top_items) - 20} more")

    # Find image files (sample up to 1000)
    print("\nSearching for image files (sampling up to 1000)...")
    extensions = Counter()
    sample_files = []
    for i, f in enumerate(base.rglob("*")):
        if i > 50_000:
            break
        if f.is_file():
            extensions[f.suffix.lower()] += 1
            if len(sample_files) < 5:
                sample_files.append(str(f.relative_to(base)))

    print(f"\nFile extensions found:")
    for ext, count in extensions.most_common(10):
        print(f"  {ext:10s}  {count:,}")

    # Infer image format
    image_exts = {
        ".dicom": "DICOM",
        ".dcm": "DICOM",
        ".jpg": "JPEG",
        ".jpeg": "JPEG",
        ".png": "PNG",
        ".nii": "NIfTI",
        ".nii.gz": "NIfTI",
        ".npz": "NPZ",
        ".npy": "NPZ",
    }
    detected_format = None
    for ext, count in extensions.most_common(3):
        if ext in image_exts:
            detected_format = image_exts[ext]
            break
    print(f"\nInferred image format: {detected_format or 'UNKNOWN'}")

    print(f"\nSample file paths (relative to base_image_dir):")
    for p in sample_files:
        print(f"  {p}")

    # Discover companion CSVs near base_image_dir
    search_dirs = [base, base.parent]
    companion_csvs = []
    for d in search_dirs:
        for f in sorted(d.glob("*.csv*")):
            companion_csvs.append(str(f))
    if companion_csvs:
        print(f"\nCompanion CSVs found near base_image_dir:")
        for c in companion_csvs:
            print(f"  {c}")

    # Look for report files
    report_exts = [".txt", ".xml"]
    report_files = []
    for i, f in enumerate(base.rglob("*")):
        if i > 50_000:
            break
        if f.is_file() and f.suffix.lower() in report_exts:
            report_files.append(str(f.relative_to(base)))
            if len(report_files) >= 5:
                break
    if report_files:
        print(f"\nReport-like files found:")
        for r in report_files:
            print(f"  {r}")
else:
    print("BASE_IMAGE_DIR not set — skipping directory exploration.")

---

## 3. Inspect the Primary CSV

Read the primary metadata CSV to discover columns, data types, and sample rows.

In [ ]:
primary_df = None
if PRIMARY_CSV_PATH and os.path.exists(PRIMARY_CSV_PATH):
    primary_df = pd.read_csv(PRIMARY_CSV_PATH)
    print(f"Primary CSV: {PRIMARY_CSV_PATH}")
    print(f"Shape (first 1000 rows): {primary_df.shape}")
    print(f"\nColumns ({len(primary_df.columns)}):")
    for col in primary_df.columns:
        dtype = primary_df[col].dtype
        nunique = primary_df[col].nunique()
        nulls = primary_df[col].isna().sum()
        sample = (
            primary_df[col].dropna().iloc[0]
            if not primary_df[col].dropna().empty
            else "N/A"
        )
        print(
            f"  {col:40s}  dtype={str(dtype):10s}  unique={nunique:>6,}  nulls={nulls:>4}  sample={sample}"
        )
    print(f"\nFirst 5 rows:")
    display(primary_df.head())
else:
    print("PRIMARY_CSV_PATH not set or file not found — skipping.")

---

## 4. Identify Patient ID, Study ID, and Image Path

Based on the columns above, identify which columns map to `patient_id`, `study_id`,
and `image_path`. If they need derivation (e.g. extracting from a path string), describe how.

In [ ]:
# >>> EDIT THIS — patient ID
PATIENT_ID_COLUMN = "Patient ID"  # column name, or "" if derived
PATIENT_ID_DERIVATION = ""

# >>> EDIT THIS — study ID
STUDY_ID_COLUMN = "Follow-up #"  # column name, or "" if derived
STUDY_ID_DERIVATION = "need to combine this with patient id to make it unique."  # e.g. "combine patient + study folder from Path"

# >>> EDIT THIS — image path
# image_path must be RELATIVE to BASE_IMAGE_DIR (the dataset class joins
# them at sample-load time). Do NOT bake a hardcoded subdirectory prefix
# into image_path that could be folded into BASE_IMAGE_DIR instead.
IMAGE_PATH_COLUMN = (
    "Image Index"  # column name containing file paths, or "" if constructed
)
IMAGE_PATH_DERIVATION = "need to derive the path through searching maybe, since this index does not show the parent folders.'"  # e.g. "column 'Image Index' contains the filename"
# prefix_to_strip: leading prefix the upstream CSV uses but that you want
# folded into BASE_IMAGE_DIR instead (e.g. "files/" for MIMIC-CXR,
# "CheXpert-v1.0/train/" for CheXpert). Keeps image_path aligned with
# BASE_IMAGE_DIR being at the deepest stable level.
IMAGE_PATH_PREFIX_TO_STRIP = ""

In [ ]:
# Validate: spot-check that BASE_IMAGE_DIR + image_path resolves to real files.
# If any sample is MISSING, either (a) BASE_IMAGE_DIR is wrong (try moving it
# deeper or stripping a prefix from image_path), or (b) the dataset has a
# structural exception — images live in sibling subdirs (like ChestX-ray14's
# images_001/ … images_012/) and the harmonizer needs to scan and prepend
# the subdir name into image_path.
if primary_df is not None and IMAGE_PATH_COLUMN and BASE_IMAGE_DIR:
    sample_paths = primary_df[IMAGE_PATH_COLUMN].dropna().head(5)
    print("Spot-checking image paths (BASE_IMAGE_DIR + image_path):")
    n_missing = 0
    for p in sample_paths:
        rel = (
            p.removeprefix(IMAGE_PATH_PREFIX_TO_STRIP)
            if IMAGE_PATH_PREFIX_TO_STRIP
            else p
        )
        full = os.path.join(BASE_IMAGE_DIR, rel)
        exists = os.path.exists(full)
        n_missing += 0 if exists else 1
        print(f"  [{'OK' if exists else 'MISSING'}] {full}")
    if n_missing:
        print(
            f"\n{n_missing}/5 samples missing. Try one of:\n"
            "  1. Recursive search to locate one sample — it tells you the\n"
            "     real subtree layout and where BASE_IMAGE_DIR should sit.\n"
            "  2. If images are split across sibling subdirs at the same\n"
            "     level (structural exception), keep BASE_IMAGE_DIR at the\n"
            "     parent and have _build_image_path scan and prepend the\n"
            "     subdir name into image_path."
        )
        # Quick recursive locate for the first sample (limited search).
        first = sample_paths.iloc[0]
        target_name = os.path.basename(
            first.removeprefix(IMAGE_PATH_PREFIX_TO_STRIP)
            if IMAGE_PATH_PREFIX_TO_STRIP
            else first
        )
        from pathlib import Path

        base = Path(BASE_IMAGE_DIR)
        for i, found in enumerate(base.rglob(target_name)):
            print(f"\n  Found {target_name} at: {found}")
            print(f"  → relative to BASE_IMAGE_DIR: {found.relative_to(base)}")
            if i >= 0:
                break
        else:
            print(f"\n  {target_name} not found anywhere under BASE_IMAGE_DIR.")
elif primary_df is not None:
    print("IMAGE_PATH_COLUMN not set — skipping path validation.")
    print("Columns available:", list(primary_df.columns))

---

## 5. Inspect the Label CSV

If labels are in a separate CSV, inspect it here. Otherwise this section analyzes
label columns from the primary CSV.

In [ ]:
label_df = None
if LABEL_CSV_PATH and os.path.exists(LABEL_CSV_PATH):
    label_df = pd.read_csv(LABEL_CSV_PATH, nrows=1000)
    print(f"Label CSV: {LABEL_CSV_PATH}")
    print(f"Shape (first 1000 rows): {label_df.shape}")
    print(f"\nColumns ({len(label_df.columns)}):")
    for col in label_df.columns:
        dtype = label_df[col].dtype
        nunique = label_df[col].nunique()
        nulls = label_df[col].isna().sum()
        print(
            f"  {col:40s}  dtype={str(dtype):10s}  unique={nunique:>6,}  nulls={nulls:>4}"
        )
    display(label_df.head())
elif LABEL_CSV_PATH:
    print(f"Label CSV not found: {LABEL_CSV_PATH}")
else:
    print("No separate label CSV — labels assumed to be in primary CSV.")
    if primary_df is not None:
        # Guess label columns: numeric columns that aren't IDs
        id_like = {"subject_id", "patient_id", "study_id", "dicom_id", "index"}
        numeric_cols = [
            c
            for c in primary_df.select_dtypes(include="number").columns
            if c.lower() not in id_like
        ]
        if numeric_cols:
            print(f"\nCandidate label columns (numeric, non-ID):")
            for col in numeric_cols:
                vals = primary_df[col].dropna().unique()
                print(f"  {col:40s}  unique_values={sorted(vals)[:10]}")

In [ ]:
# >>> EDIT THIS — label configuration
LABEL_COLUMNS = [
    "Atelectasis",
    "Consolidation",
    "Infiltration",
    "Pneumothorax",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Effusion",
    "Pneumonia",
    "Pleural_thickening",
    "Cardiomegaly",
    "Nodule Mass",
    "Hernia",
    "No findings",
    # List label column names exactly as they appear in the CSV, sorted alphabetically.
    # e.g. "Atelectasis", "Cardiomegaly", ...
]

# Join columns if labels are in a separate CSV
LABEL_JOIN_COLUMNS = []  # e.g. ["subject_id", "study_id"]

# Uncertain label handling
NAN_ACTION = "to_zero"  # "keep" or "to_zero"
NEGATIVE_ONE_ACTION = "to_nan"  # "keep", "to_nan", or "drop"
DROP_UNCERTAIN_ROWS = True  # drop rows with NaN after mapping?

In [ ]:
# Analyze label value distributions
check_df = label_df if label_df is not None else primary_df
if check_df is not None and LABEL_COLUMNS:
    present = [c for c in LABEL_COLUMNS if c in check_df.columns]
    missing = [c for c in LABEL_COLUMNS if c not in check_df.columns]
    if missing:
        print(f"WARNING: columns not found in CSV: {missing}")
    if present:
        print("Label value distributions:")
        for col in present:
            vc = check_df[col].value_counts(dropna=False).head(10)
            print(f"\n  {col}:")
            for val, count in vc.items():
                print(f"    {str(val):>8s}  {count:>6,}")
elif check_df is not None:
    print("LABEL_COLUMNS not set — fill them in above.")

---

## 6. View Position, Masks, Bboxes, Reports

Inspect optional features. Skip sections that don't apply.

In [ ]:
# >>> EDIT THIS — view position
VIEW_POSITION_COLUMN = (
    "View Position"  # CSV column name, or "" if none / from DICOM headers
)
VIEW_POSITION_SOURCE = ""  # "csv", "dicom_header", or ""

# >>> EDIT THIS — masks
MASKS_SUPPORTED = False
MASK_COLUMN = ""  # column name, or ""
MASK_FORMAT = ""  # "rle", "file_path", or ""

# >>> EDIT THIS — bounding boxes
BBOXES_SUPPORTED = True
BBOX_COLUMN = "Bbox x, Bbox y, Bbox w, Bbox h"

# >>> EDIT THIS — reports
REPORTS_SUPPORTED = False
REPORT_FORMAT = ""  # "text_files", "csv_column", or ""
REPORT_PATH_COLUMN = ""  # column in report CSV with file paths
REPORT_BASE_DIR = ""  # directory containing report files

In [ ]:
# Inspect optional CSVs
for name, path in [
    ("View Position", VIEW_POSITION_CSV_PATH),
    ("Report", REPORT_CSV_PATH),
    ("Mask", MASK_CSV_PATH),
    ("Bbox", BBOX_CSV_PATH),
]:
    if path and os.path.exists(path):
        df_opt = pd.read_csv(path, nrows=5)
        print(f"\n{name} CSV: {path}")
        print(f"  Columns: {list(df_opt.columns)}")
        display(df_opt)
    elif path:
        print(f"\n{name} CSV not found: {path}")

# If DICOM, try reading ViewPosition from a sample header
if (
    detected_format if "detected_format" in dir() else None
) == "DICOM" and sample_files:
    try:
        import pydicom

        sample_dcm = os.path.join(BASE_IMAGE_DIR, sample_files[0])
        dcm = pydicom.dcmread(sample_dcm, stop_before_pixels=True)
        vp = getattr(dcm, "ViewPosition", None)
        print(f"\nDICOM header sample ({sample_files[0]}):")
        print(f"  ViewPosition: {vp}")
        print(f"  Modality:     {getattr(dcm, 'Modality', 'N/A')}")
        print(
            f"  Rows x Cols:  {getattr(dcm, 'Rows', '?')} x {getattr(dcm, 'Columns', '?')}"
        )
    except Exception as e:
        print(f"\nCould not read DICOM header: {e}")

---

## 7. Extra Parameters and Preprocessing

Any additional constructor parameters or custom preprocessing steps.

In [ ]:
# >>> EDIT THIS — extra init parameters (dict of name -> description)
# e.g. {"dicom_base_dir": "root of DICOM tree for reading ViewPosition from headers"}
EXTRA_INIT_PARAMS = {}

# >>> EDIT THIS — HU window for 3D CT datasets, or None
HU_WINDOW = None  # e.g. [-1000, 400]

# >>> EDIT THIS — description of any custom preprocessing, or ""
CUSTOM_PREPROCESSING = ""  # e.g. "melt/pivot label columns, glob DICOM files"

---

## 8. Filename Variants

List all known filename variants for each CSV so that `infer_path` can auto-discover
them regardless of which version the user has on disk (e.g. `.csv` vs `.csv.gz`).

In [ ]:
# >>> EDIT THIS — all known filename variants for each CSV
# Include .csv AND .csv.gz variants, legacy names, etc.
PRIMARY_CSV_VARIANTS = (
    []
)  # e.g. ["cxr-record-list.csv.gz", "cxr-record-list.csv", "mimic-cxr-2.0.0-metadata.csv"]
LABEL_CSV_VARIANTS = (
    []
)  # e.g. ["mimic-cxr-2.0.0-chexpert.csv", "mimic-cxr-2.0.0-chexpert.csv.gz"]
REPORT_CSV_VARIANTS = []  # e.g. ["cxr-study-list.csv.gz", "cxr-study-list.csv"]
VIEW_POSITION_CSV_VARIANTS = []
MASK_CSV_VARIANTS = []
BBOX_CSV_VARIANTS = []

# Auto-suggest variants from provided paths
for label, path, variants in [
    ("Primary CSV", PRIMARY_CSV_PATH, PRIMARY_CSV_VARIANTS),
    ("Label CSV", LABEL_CSV_PATH, LABEL_CSV_VARIANTS),
    ("Report CSV", REPORT_CSV_PATH, REPORT_CSV_VARIANTS),
]:
    if path and not variants:
        name = os.path.basename(path)
        suggestions = [name]
        if name.endswith(".gz"):
            suggestions.append(name.removesuffix(".gz"))
        elif name.endswith(".csv"):
            suggestions.append(name + ".gz")
        print(f"  {label}: auto-suggested variants = {suggestions}")
        print(f"    (add these to the corresponding *_VARIANTS list above)")

---

## 9. Build and Export JSON Config

Assembles all information into the `configs/datasets/<registry_key>.json` file
used by the `add-dataset` skill.

In [ ]:
config = {
    "_comment": DESCRIPTION or None,
    "dataset_name": DATASET_NAME or None,
    "registry_key": REGISTRY_KEY or None,
    "modality": MODALITY or None,
    "image_format": (
        detected_format if "detected_format" in dir() and detected_format else None
    ),
    "base_image_dir": BASE_IMAGE_DIR or None,
    "primary_csv": {
        "_comment": "The main metadata CSV that lists every image row.",
        "path": PRIMARY_CSV_PATH or None,
        "filename_variants": PRIMARY_CSV_VARIANTS or [],
    },
    "patient_id": {
        "_comment": "How to obtain the patient ID. Provide column if it exists, or describe derivation.",
        "column": PATIENT_ID_COLUMN or None,
        "derivation": PATIENT_ID_DERIVATION or None,
    },
    "study_id": {
        "_comment": "How to obtain the study ID. Provide column if it exists, or describe derivation.",
        "column": STUDY_ID_COLUMN or None,
        "derivation": STUDY_ID_DERIVATION or None,
    },
    "image_path": {
        "_comment": "How to obtain the image file path relative to base_image_dir.",
        "column": IMAGE_PATH_COLUMN or None,
        "derivation": IMAGE_PATH_DERIVATION or None,
        "prefix_to_strip": IMAGE_PATH_PREFIX_TO_STRIP or None,
    },
    "labels": {
        "_comment": "Label columns and how to handle uncertain values.",
        "columns": sorted(LABEL_COLUMNS),
        "separate_csv": {
            "_comment": "If labels are in a separate CSV, describe path and how to join.",
            "path": LABEL_CSV_PATH or None,
            "filename_variants": LABEL_CSV_VARIANTS or [],
            "join_columns": LABEL_JOIN_COLUMNS or [],
        },
        "uncertain_handling": {
            "_comment": "How to handle missing or uncertain label values.",
            "nan_action": NAN_ACTION,
            "negative_one_action": NEGATIVE_ONE_ACTION,
            "drop_uncertain_rows": DROP_UNCERTAIN_ROWS,
        },
    },
    "view_position": {
        "_comment": "View position info (e.g. AP/PA/Lateral). Can come from CSV or DICOM headers.",
        "column": VIEW_POSITION_COLUMN or None,
        "source": VIEW_POSITION_SOURCE or None,
        "separate_csv": {
            "path": VIEW_POSITION_CSV_PATH or None,
            "filename_variants": VIEW_POSITION_CSV_VARIANTS or [],
            "join_columns": [],
        },
    },
    "masks": {
        "_comment": "Segmentation mask support.",
        "supported": MASKS_SUPPORTED,
        "column": MASK_COLUMN or None,
        "format": MASK_FORMAT or None,
    },
    "bboxes": {
        "_comment": "Bounding box support.",
        "supported": BBOXES_SUPPORTED,
        "column": BBOX_COLUMN or None,
    },
    "reports": {
        "_comment": "Radiology report support. Describe format and location.",
        "supported": REPORTS_SUPPORTED,
        "format": REPORT_FORMAT or None,
        "separate_csv": {
            "_comment": "CSV that maps studies to report file paths.",
            "path": REPORT_CSV_PATH or None,
            "filename_variants": REPORT_CSV_VARIANTS or [],
            "join_columns": [],
        },
        "base_dir": REPORT_BASE_DIR or None,
        "path_column": REPORT_PATH_COLUMN or None,
    },
    "extra_init_params": {
        "_comment": "Any additional constructor parameters not covered above.",
        **(EXTRA_INIT_PARAMS or {}),
    },
    "hu_window": HU_WINDOW,
    "custom_preprocessing": CUSTOM_PREPROCESSING or None,
}

# Pretty-print for review
print(json.dumps(config, indent=2))

### Completeness Check

Verify that all required fields are filled in before saving.

In [ ]:
# Completeness check
required = {
    "dataset_name": config["dataset_name"],
    "registry_key": config["registry_key"],
    "modality": config["modality"],
    "image_format": config["image_format"],
    "base_image_dir": config["base_image_dir"],
    "primary_csv.path": config["primary_csv"]["path"],
    "patient_id (column or derivation)": config["patient_id"]["column"]
    or config["patient_id"]["derivation"],
    "study_id (column or derivation)": config["study_id"]["column"]
    or config["study_id"]["derivation"],
    "image_path (column or derivation)": config["image_path"]["column"]
    or config["image_path"]["derivation"],
    "labels.columns": config["labels"]["columns"],
}

missing = [k for k, v in required.items() if not v]
if missing:
    print("MISSING required fields (go back and fill these in):")
    for m in missing:
        print(f"  - {m}")
else:
    print("All required fields are filled in.")

### Save to File

Write the JSON config to `configs/datasets/<registry_key>.json`.

In [ ]:
assert not missing, f"Cannot save — missing required fields: {missing}"
assert REGISTRY_KEY, "REGISTRY_KEY must be set before saving."

output_dir = Path("../configs/datasets")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{REGISTRY_KEY}.json"

with open(output_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config saved to: {output_path.resolve()}")
print(f"\nNext step: run  /add-dataset  and point it to this JSON file.")